In [4]:
# Part 2: Embeddings
import torch.nn as nn
#Task 1.07: Bag-of-words classifier
"""
How the bag-of-words classifier works:

  Input: a batch of reviews, each represented as a list of token IDs
         Shape: (batch_size, seq_len)

  Step 1 — Embedding lookup:
    self.embedding(x) looks up a vector for every token ID.
    Output shape: (batch_size, seq_len, embedding_dim)

  Step 2 — Mean pooling (.mean(dim=-2)):
    Average all token vectors in the review into one vector.
    dim=-2 means "average along the sequence (token) dimension".
    Output shape: (batch_size, embedding_dim)
    This is the "bag" — word ORDER is lost, only presence/frequency matters.

  Step 3 — Linear classification:
    self.linear maps the embedding vector to class scores.
    Output shape: (batch_size, num_classes)

Why only ONE nn.Embedding when the diagram shows three?
  The diagram shows separate embedding layers for each word POSITION,
  but in a bag-of-words model ALL tokens share the same embedding table.
  One nn.Embedding holds all token vectors; each token ID indexes into it.

What does dim=-2 do?
  Tensors are indexed from the last dimension backwards with negative indices.
  Shape is (batch, seq_len, embed_dim) = dims 0, 1, 2 = dims 0, -2, -1.
  dim=-2 averages over the seq_len dimension → collapses sequences to one vector.
"""


class Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))


In [2]:
#Dataset
from torch.utils.data import Dataset


class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        with open(filename) as f:
            tokenized_lines = [line.split() for line in f]
        self.items = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[list[str], str]:
        return self.items[idx]

In [5]:
#Task 1.08: Vectoriser
from collections import Counter

import torch

# Type abbreviation for review-label pairs
type Item = tuple[list[str], str]


class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # zip(*dataset) transposes the list of (review, label) pairs:
        #   reviews: tuple[list[str], ...]  -- one token list per example
        #   labels:  tuple[str, ...]        -- one label string per example
        reviews, labels = zip(*dataset)

        # Count the tokens and get the most common ones
        counter = Counter(t for r in reviews for t in r)
        most_common = [t for t, _ in counter.most_common(n_vocab - 2)]

        # Token-to-ID mapping: PAD=0, UNK=1, then most common tokens
        self.t2i = {t: i for i, t in enumerate([self.PAD, self.UNK] + most_common)}
        # Label-to-ID mapping: sorted alphabetically for determinism
        self.l2i = {l: i for i, l in enumerate(sorted(set(labels)))}

    def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Convert a batch of (review, label) pairs into tensors.

        Returns
        -------
        X : torch.Tensor, shape (m, max_len), dtype long
            Matrix of token IDs; shorter reviews are right-padded with PAD (index 0).
        y : torch.Tensor, shape (m,), dtype long
            Vector of label IDs.
        """
        reviews, labels = zip(*items)

        # Convert labels to integer IDs
        y = torch.tensor([self.l2i[label] for label in labels], dtype=torch.long)

        # Convert reviews to padded token-ID sequences
        unk_id = self.t2i[self.UNK]
        pad_id = self.t2i[self.PAD]  # always 0
        max_len = max(len(r) for r in reviews)

        rows = []
        for review in reviews:
            # Map tokens to IDs (unknown tokens -> UNK)
            ids = [self.t2i.get(token, unk_id) for token in review]
            # Pad to max_len so all rows have the same length
            ids += [pad_id] * (max_len - len(ids))
            rows.append(ids)

        X = torch.tensor(rows, dtype=torch.long)
        return X, y

In [6]:
#Task 1.09: Training loop
import torch.nn.functional as F


def train(
    filename: str = "reviews-train.txt",  # Path to training data
    label: int = 0,                        # 0=product category, 1=sentiment
    n_vocab: int = 1024,                   # Vocabulary size for the vectoriser
    embedding_dim: int = 64,               # Dimensionality of embedding space
    lr: float = 0.001,                     # Adam learning rate
    batch_size: int = 16,                  # Examples per gradient update
    n_epochs: int = 10,                    # Total passes through data
):
    # 1. Load the training data and choose which label to predict
    dataset = ReviewDataset(filename, label=label)

    # 2. Build the vectoriser: learns vocabulary from the training set
    processor = ReviewVectorizer(dataset, n_vocab)

    # 3. Instantiate the bag-of-words classifier
    model = Classifier(n_vocab, embedding_dim, len(processor.l2i))

    # 4. Adam optimiser (adaptive learning rates per parameter)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # 5. DataLoader: batches + shuffles data; collate_fn converts each batch
    #    of (review, label) pairs into (X, y) tensors via ReviewVectorizer.__call__
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=processor,
    )

    for epoch in range(n_epochs):
        model.train()       # Training mode (enables dropout/BN train behaviour)
        running_loss = 0

        for bx, by in data_loader:
            optimizer.zero_grad()           # 6. Clear gradients from previous step
            output = model(bx)              # 7. Forward pass: raw logit scores
            loss = F.cross_entropy(output, by)  # 8. Cross-entropy loss
            loss.backward()                 # 9. Backpropagation: compute gradients
            optimizer.step()                # 10. Update model parameters
            running_loss += loss.item()

        print(f"Epoch {epoch}, loss: {running_loss / len(data_loader):.4f}")

    return processor, model

In [7]:
#Task 1.10: Training the classifier
import torch

# --- Product category classifier (camera vs. music) ---
print("=== Training: Product Category (camera vs. music) ===")
torch.manual_seed(42)  # Fix seed for reproducibility
vectorizer_cat, model_cat = train(
    filename="reviews-train.txt",
    label=0,
)

=== Training: Product Category (camera vs. music) ===
Epoch 0, loss: 0.6838
Epoch 1, loss: 0.6486
Epoch 2, loss: 0.6046
Epoch 3, loss: 0.5415
Epoch 4, loss: 0.4708
Epoch 5, loss: 0.4008
Epoch 6, loss: 0.3403
Epoch 7, loss: 0.2900
Epoch 8, loss: 0.2518
Epoch 9, loss: 0.2230


In [8]:
# --- Sentiment classifier (neg vs. pos) ---
print("=== Training: Sentiment (neg vs. pos) ===")
torch.manual_seed(42)
vectorizer_sent, model_sent = train(
    filename="reviews-train.txt",
    label=1,
)

=== Training: Sentiment (neg vs. pos) ===
Epoch 0, loss: 0.6932
Epoch 1, loss: 0.6878
Epoch 2, loss: 0.6784
Epoch 3, loss: 0.6719
Epoch 4, loss: 0.6615
Epoch 5, loss: 0.6488
Epoch 6, loss: 0.6327
Epoch 7, loss: 0.6122
Epoch 8, loss: 0.5928
Epoch 9, loss: 0.5724


In [9]:
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
):
    i2t = {i: t for t, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().numpy()
    items = [(i2t[i], e) for i, e in enumerate(embeddings)]
    with open(vectors_filename, "wt") as f1, open(metadata_filename, "wt") as f2:
        for w, e in items:
            print("\t".join("{:.5f}".format(x) for x in e), file=f1)
            print(w, file=f2)

# Save embeddings for both classifiers
save_embeddings(vectorizer_cat, model_cat, "vectors_cat.tsv", "metadata_cat.tsv")
save_embeddings(vectorizer_sent, model_sent, "vectors_sent.tsv", "metadata_sent.tsv")
print("Embedding files saved.")
print("Load each pair into http://projector.tensorflow.org to explore the vector space.")

Embedding files saved.
Load each pair into http://projector.tensorflow.org to explore the vector space.


In [ ]:
#TASK 1.11 — Inspecting the embeddings
"""
How to use the Embedding Projector:
  1. Go to https://projector.tensorflow.org/
  2. Click "Load data" (top left)
  3. Upload vectors.tsv  as "Tensor"
  4. Upload metadata.tsv as "Metadata"

What to expect:

  Category classifier (camera vs. music):
    Two clearly separated clusters — camera words (lens, zoom, battery,
    megapixel) cluster together, music words (album, track, song, bass)
    cluster together. The separation should be very clean.

  Sentiment classifier (pos vs. neg):
    Clusters based on emotional valence — positive words (great, love,
    excellent, perfect) in one region; negative words (waste, terrible,
    broken, disappointed) in another. More overlap than category.

  repair vs. sturdy:
    In the CATEGORY model: both appear near "camera" vocabulary (repairs
    and sturdiness are common camera concerns) → CLOSE together.
    In the SENTIMENT model: "repair" suggests something broke → negative;
    "sturdy" suggests quality → positive → FARTHER apart.
    This shows the embeddings encode task-relevant relationships.

Dimensionality reduction:
  PCA:   Fast, global structure, good for seeing overall shape of clusters.
  T-SNE: Better at revealing local clusters, but can distort global distances.
  UMAP:  Best balance of local and global structure; often most interpretable.
"""

In [10]:
#Task 1.12: Initialisation of embedding layers

import math
import torch.nn.init as init


class ClassifierKaiming(nn.Module):
    """
    Same architecture as Classifier but with Kaiming uniform initialisation
    for the embedding weights (matching the default init of nn.Linear).
    """

    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)
        # Apply Kaiming uniform init to the embedding weight matrix.
        # a=sqrt(5) matches the leaky-ReLU slope used in nn.Linear's default.
        init.kaiming_uniform_(self.embedding.weight, a=math.sqrt(5))

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))


def train_kaiming(
    filename: str = "reviews-train.txt",
    label: int = 0,
    n_vocab: int = 1024,
    embedding_dim: int = 64,
    lr: float = 0.001,
    batch_size: int = 16,
    n_epochs: int = 10,
):
    """Same as train() but uses ClassifierKaiming."""
    dataset = ReviewDataset(filename, label=label)
    processor = ReviewVectorizer(dataset, n_vocab)
    model = ClassifierKaiming(n_vocab, embedding_dim, len(processor.l2i))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True, collate_fn=processor
    )
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0
        for bx, by in data_loader:
            optimizer.zero_grad()
            output = model(bx)
            loss = F.cross_entropy(output, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch}, loss: {running_loss / len(data_loader):.4f}")
    return processor, model


# Train with Kaiming init and compare against default initialisation
print("=== Kaiming Init: Category classifier ===")
torch.manual_seed(42)
vectorizer_cat_k, model_cat_k = train_kaiming(label=0)

print("\n=== Kaiming Init: Sentiment classifier ===")
torch.manual_seed(42)
vectorizer_sent_k, model_sent_k = train_kaiming(label=1)


=== Kaiming Init: Category classifier ===
Epoch 0, loss: 0.6775
Epoch 1, loss: 0.5864
Epoch 2, loss: 0.4349
Epoch 3, loss: 0.3134
Epoch 4, loss: 0.2375
Epoch 5, loss: 0.1900
Epoch 6, loss: 0.1585
Epoch 7, loss: 0.1328
Epoch 8, loss: 0.1168
Epoch 9, loss: 0.1017

=== Kaiming Init: Sentiment classifier ===
Epoch 0, loss: 0.6915
Epoch 1, loss: 0.6817
Epoch 2, loss: 0.6580
Epoch 3, loss: 0.6171
Epoch 4, loss: 0.5678
Epoch 5, loss: 0.5207
Epoch 6, loss: 0.4774
Epoch 7, loss: 0.4402
Epoch 8, loss: 0.4106
Epoch 9, loss: 0.3862
